# Tugas #4: Data-Centric Improvement and Model Interpretation

Notebook ini melanjutkan pipeline terbaik Tugas #3. Model dan preprocessing **tidak diubah**:
yang diubah hanya data train (imbalance treatment, review label Cleanlab), supaya selisih
performa bisa diatributkan ke perubahan data.

Setup yang dipertahankan dari Tugas #3:
- Split raw 70/30 stratified (`random_state=42`). Semua CV hanya di data train.
- `StratifiedKFold(5, shuffle=True, random_state=42)`, fold identik di semua eksperimen.
- Primary metric: **MCC**. Macro F1 ditambahkan sebagai kolom pendamping.
- Test set baru dipakai sekali di evaluasi akhir.

In [1]:
import sys, os, warnings
sys.path.append('../src')
warnings.filterwarnings('ignore', category=FutureWarning)

import numpy as np
import pandas as pd

from data_utils import load_data
from preprocessing import split_raw
from models import get_models
from pipeline_experiments import build_pipeline, CHI2_K
from data_centric import cv_evaluate, get_cv, PRIMARY_METRIC

pd.set_option('display.width', 140)
TABLE_DIR, FIG_DIR = '../results/tables', '../results/figures'
os.makedirs(TABLE_DIR, exist_ok=True); os.makedirs(FIG_DIR, exist_ok=True)

df = load_data()
X_train_raw, X_test_raw, y_train, y_test = split_raw(df)
X, y = X_train_raw, y_train   # alias: semua CV di bawah HANYA memakai data train

print('Train:', X_train_raw.shape, '| proporsi positif:', round(y_train.mean(), 4))
print('Test :', X_test_raw.shape, '| proporsi positif:', round(y_test.mean(), 4), '(tidak disentuh)')

Train: (8631, 17) | proporsi positif: 0.1548
Test : (3699, 17) | proporsi positif: 0.1546 (tidak disentuh)


## 1. Baseline (D0)

D0 adalah pipeline terbaik Tugas #3 apa adanya: `CorrectedPreprocessor` (one-hot `Month` dan
`VisitorType`, chi-square top-20, standard scaling) lalu SMOTE 1:1 di dalam pipeline, lalu
XGBoost `max_depth=2`. Di Tugas #3 pipeline ini tetap yang terbaik di test set (MCC 0,5955),
mengalahkan versi log-transform/pruning dan versi tuned.

Sebagai sanity check, MCC CV harus sama dengan Tugas #3 (0,6257).

In [2]:
xgb = get_models()['XGBoost']
pipe_d0 = build_pipeline(xgb, k=CHI2_K, drop_features=None, log_transform=False, use_smote=True)

summary_d0, folds_d0 = cv_evaluate(pipe_d0, X, y)
print('D0 (baseline Tugas #3), 5-fold CV di data train:')
print(summary_d0)

T3_MCC = 0.6257
assert abs(summary_d0.loc['MCC', 'Mean'] - T3_MCC) < 1e-4, 'Baseline tidak cocok dengan Tugas #3'
print(f'\nSanity check OK: MCC = {summary_d0.loc["MCC", "Mean"]:.4f} (Tugas #3: {T3_MCC})')

D0 (baseline Tugas #3), 5-fold CV di data train:
                        Mean     Std
Accuracy              0.8940  0.0069
Precision             0.6343  0.0217
TPR (Recall)          0.7470  0.0276
F1-Score              0.6857  0.0190
TNR                   0.9209  0.0075
MCC                   0.6257  0.0228
auROC                 0.9265  0.0076
auPR                  0.7307  0.0233
Precision (weighted)  0.9029  0.0060
F1-Score (weighted)   0.8974  0.0064
Macro F1              0.8110  0.0115

Sanity check OK: MCC = 0.6257 (Tugas #3: 0.6257)


In [3]:
print('Skor per fold D0:')
print(folds_d0[['MCC', 'Macro F1', 'F1-Score', 'TPR (Recall)', 'Precision', 'auPR']].round(4))

summary_d0.to_csv(f'{TABLE_DIR}/tugas4_d0_baseline_cv.csv')
folds_d0.round(4).to_csv(f'{TABLE_DIR}/tugas4_d0_baseline_folds.csv')

Skor per fold D0:
           MCC  Macro F1  F1-Score  TPR (Recall)  Precision    auPR
Fold 1  0.6147    0.8055    0.6769        0.7388     0.6246  0.7032
Fold 2  0.6707    0.8338    0.7232        0.7828     0.6720  0.7601
Fold 3  0.6165    0.8048    0.6777        0.7640     0.6090  0.7479
Fold 4  0.6191    0.8073    0.6803        0.7491     0.6231  0.7390
Fold 5  0.6076    0.8033    0.6703        0.7004     0.6426  0.7035


Baseline cocok dengan Tugas #3, jadi fold dan pipeline terbukti identik. Std MCC
antar fold sekitar 0,02: selisih antar eksperimen data di bawah ukuran ini perlu dibaca sebagai
noise, bukan perbaikan.